# Phase 4.6: export the real Llama-3.2-1B .pte with XNNPACK delegation (Kaggle, 30GB RAM)

Runs `export_llama.py --real-weights` with `XnnpackDynamicallyQuantizedPartitioner` enabled --
the same script already validated locally (Phase 4.4/4.5), now trying to clear a memory wall
hit on the local machine: XNNPACK's own weight-serialization code hashes every weight tensor
(`sha256(bytes(weight))`, for dedup) during `to_edge_transform_and_lower`, and on top of
everything `torch.export`/`to_edge` already hold in memory for a real ~1.24B-param model,
that pushed past available RAM locally -- 8 layers worked, 10 hit a clean `MemoryError`
inside XNNPACK's `node_visitor.py`, 12+ segfaulted (same exhaustion, harder failure mode).
Kaggle's 30GB should clear this with room to spare.

**Before running:** in the notebook settings (right sidebar), turn ON "Internet" -- Kaggle
kernels default to no internet access, which will break both the pip installs and the model
download.

Needs a HF token with access to the gated `meta-llama/Llama-3.2-1B` repo.

In [ ]:
!pip install -q torch==2.12.0 torchvision torchaudio executorch==1.3.1 transformers pybind11 huggingface_hub hf_transfer
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
from huggingface_hub import login
login()  # paste an HF token with access to meta-llama/Llama-3.2-1B when prompted

In [ ]:
!rm -rf /kaggle/working/windowed-attention-cpu
!git clone --depth 1 https://github.com/parzi-val/windowed-attention-cpu.git /kaggle/working/windowed-attention-cpu
%cd /kaggle/working/windowed-attention-cpu/executorch/phase4_llama

In [ ]:
# Build the pybind kernel used for export_llama.py's eager sanity check (the actual .pte doesn't
# embed this -- pure framework-free C++, no ExecuTorch dependency, so a plain g++ compile
# against the Phase 3 kernel source is enough).
import sysconfig
ext_suffix = sysconfig.get_config_var("EXT_SUFFIX")

!mkdir -p ../phase3_kv_cache/build
!g++ -O2 -shared -std=c++17 -fPIC $(python3 -m pybind11 --includes) \
    ../phase3_kv_cache/windowed_sdpa_kv_cache_kernel.cpp \
    ../phase3_kv_cache/windowed_sdpa_kv_cache_pybind.cpp \
    -o ../phase3_kv_cache/build/windowed_sdpa_kv_cache_pybind{ext_suffix}
!ls ../phase3_kv_cache/build/

In [ ]:
# Sanity check: does this environment's pip-installed executorch actually bundle a working
# flatc (the local Windows issue), or do we also need FLATC_EXECUTABLE set here? export_llama.py
# now checks this itself and raises a clear error if neither works, so this cell is just an
# early, cheap heads-up before the ~5-minute real-weight export run below.
from executorch.exir._serialize._flatbuffer import _get_flatc_path
import os, shutil
p = _get_flatc_path()
print(f"resolved flatc path: {p}")
print(f"usable: {os.path.isfile(p) or bool(shutil.which(p))}")

In [ ]:
!python export_llama.py --arm map50 --real-weights

In [ ]:
# Download the result -- Kaggle doesn't have Colab's google.colab.files helper, so zip it and
# use the Output/Data pane's download button, or grab it directly from /kaggle/working/.
!ls -la llama_3.2_1b_map50.pte
print("\nFile is at /kaggle/working/windowed-attention-cpu/executorch/phase4_llama/llama_3.2_1b_map50.pte")
print("Download it via the Kaggle notebook's Output pane (top right, or File > Output).")